# Research Assist · Worker LLM trên Kaggle

Notebook phục vụ model sinh câu trả lời cho Research Assist đang chạy trên thiết bị của
bạn. Chọn **Accelerator → GPU** và **Internet → On** trong Kaggle.
Chọn `7b` để chạy trên một T4 hoặc GPU tương thích. Chế độ thử nghiệm `32b`
cần **GPU T4 x 2**, nạp hai phân đoạn trọng số bằng tensor parallelism.

1. Mở **Add-ons → Secrets**, tạo `RESEARCH_GPU_API_KEY` bằng một chuỗi ngẫu nhiên
   dài ít nhất 32 ký tự, rồi cho notebook truy cập secret.
2. Đặt `START_LLM_WORKER = True` trong ô cấu hình. Bật thêm
   `OPEN_PUBLIC_TUNNEL` nếu Research Assist chạy ngoài Kaggle. Chạy các ô đến **Khởi động
   worker**. Lần đầu cần tải thư viện và trọng số.
3. Khi notebook xác nhận phản hồi JSON đúng schema, sao chép **Endpoint /v1** và **Model**.
   Trong Research Assist, mở cấu hình model → **GPU Worker** → provider
   **OpenAI-compatible**. Nhập Endpoint, Model, cùng khóa đã tạo ở bước 1.
   Lưu, kiểm tra kết nối, sau đó chọn GPU cho agent muốn sử dụng.

Giữ phiên Kaggle chạy khi đang dùng worker. Link Cloudflare là tạm thời và thay đổi
mỗi lần mở lại. Khi cần đổi cấu hình, sửa `MODEL_PROFILE` và chạy lại đến ô khởi động.
Mở ô dừng riêng ở cuối khi đã dùng xong. Run All tự giữ worker chạy cho đến khi
bạn dừng ô khởi động hoặc kết thúc phiên.

- vLLM 0.10.2 dùng môi trường Python riêng và wheel CUDA 12.8, cần driver tương thích.
  Cấu hình AWQ dùng FP16 và XFormers cho Turing. Thông tin lỗi GPU và nạp model nằm
  trong log được in ở lúc khởi động. Nếu profile 32B không đủ VRAM trên phiên
  đang dùng, chọn `7b`.
- Giới hạn ngữ cảnh gồm cả đầu vào và phần sinh ra. Nếu model báo vượt giới hạn,
  giảm độ dài đầu vào hoặc số token đầu ra của agent.
- Đầu vào suy luận đi từ Research Assist qua tunnel tới GPU. Chỉ dùng dữ liệu
  bạn có thể gửi qua kết nối đó. Token đọc từ Kaggle Secrets và không in ra output.

Nguồn kỹ thuật [vLLM GPU](https://docs.vllm.ai/en/v0.10.2/getting_started/installation/gpu.html),
[vLLM Structured Outputs](https://docs.vllm.ai/en/v0.10.2/features/structured_outputs.html),
[Qwen 7B](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct-AWQ) và
[Qwen 32B](https://huggingface.co/Qwen/Qwen2.5-32B-Instruct-AWQ).

In [ ]:
START_LLM_WORKER = False
MODEL_PROFILE = "7b"  # "32b" requires two GPUs
OPEN_PUBLIC_TUNNEL = False  # Set True for Research Assist running outside Kaggle.
KEEP_WORKER_RUNNING = True   # Set False to return the control cell immediately.

In [ ]:
"""Start and stop an optional Cloudflare Quick Tunnel in an interactive notebook."""

import hashlib
import json
import os
import platform
import queue
import re
import shutil
import subprocess
import tempfile
import threading
import time
import urllib.request
from pathlib import Path


def find_cloudflared(working_dir: Path | str) -> str:
    installed = shutil.which("cloudflared")
    if installed:
        return installed
    machine = platform.machine().lower()
    if platform.system() != "Linux" or machine not in {"x86_64", "amd64", "aarch64", "arm64"}:
        raise RuntimeError("Install cloudflared for your operating system and put it on PATH.")
    arch = "amd64" if machine in {"x86_64", "amd64"} else "arm64"
    name = f"cloudflared-linux-{arch}"
    directory = Path(working_dir)
    directory.mkdir(parents=True, exist_ok=True)
    request = urllib.request.Request(
        "https://api.github.com/repos/cloudflare/cloudflared/releases/latest",
        headers={"User-Agent": "Research-Assist-Notebook"},
    )
    with urllib.request.urlopen(request, timeout=15) as response:
        release = json.load(response)
    asset = next((item for item in release["assets"] if item["name"] == name), None)
    if not asset or not re.fullmatch(r"sha256:[a-fA-F0-9]{64}", asset.get("digest") or ""):
        raise RuntimeError("No published SHA256 digest for cloudflared. Install it manually.")
    digest = asset["digest"].split(":", 1)[1].lower()
    target = directory / name
    if target.is_file():
        with target.open("rb") as existing:
            if hashlib.file_digest(existing, "sha256").hexdigest() == digest:
                target.chmod(0o700)
                return str(target.resolve())
    deadline = time.monotonic() + 90
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(dir=directory, delete=False) as output:
            temporary = Path(output.name)
            checksum = hashlib.sha256()
            with urllib.request.urlopen(asset["browser_download_url"], timeout=15) as response:
                while block := response.read(1024 * 1024):
                    if time.monotonic() >= deadline:
                        raise TimeoutError("Downloading cloudflared took too long.")
                    checksum.update(block)
                    output.write(block)
        if checksum.hexdigest() != digest:
            raise RuntimeError("cloudflared does not match the official release checksum.")
        temporary.chmod(0o700)
        os.replace(temporary, target)
        return str(target.resolve())
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)


def stop_process(process: subprocess.Popen) -> None:
    """Reap the process even if it ignores a graceful stop."""
    if process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=5)


def start_quick_tunnel(
    port: int, working_dir: Path | str, *, timeout: float = 60
) -> tuple[subprocess.Popen, str]:
    """Return the owned process and assigned HTTPS URL, or tear it down on error."""
    if not 1 <= port <= 65535 or timeout <= 0:
        raise ValueError("Invalid tunnel port or timeout.")
    executable = find_cloudflared(working_dir)
    process = subprocess.Popen(
        [executable, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    results: queue.Queue[str | None] = queue.Queue()

    def read_output() -> None:
        stream = process.stdout
        if stream is None:
            results.put(None)
            return
        found = False
        with stream:
            for line in stream:
                match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com\b", line)
                if not found and match:
                    found = True
                    results.put(match.group())
        results.put(None)

    try:
        threading.Thread(target=read_output, daemon=True).start()
        try:
            url = results.get(timeout=timeout)
        except queue.Empty:
            raise TimeoutError("No Cloudflare URL received before the timeout.") from None
        if not url or process.poll() is not None:
            raise RuntimeError("cloudflared exited before the tunnel was ready.")
        return process, url
    except BaseException:
        stop_process(process)
        raise

In [ ]:
"""Install and run the isolated vLLM worker from the standalone Kaggle notebook."""

import atexit
import json
import os
import platform
import re
import signal
import socket
import subprocess
import sys
import time
import urllib.error
import urllib.request
import venv
from pathlib import Path

VLLM_VERSION = "0.10.2"
REQUIREMENTS = [
    f"vllm=={VLLM_VERSION}",
    "transformers==4.56.1",
    "openai==1.107.2",
    "httpx==0.28.1",
]
PROFILES = {
    "7b": ("Qwen/Qwen2.5-7B-Instruct-AWQ", "b25037543e9394b818fdfca67ab2a00ecc7dd641", 1, 16384),
    "32b": ("Qwen/Qwen2.5-32B-Instruct-AWQ", "5c7cb76a268fc6cfbb9c4777eb24ba6e27f9ee6c", 2, 8192),
}


class NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None


HTTP = urllib.request.build_opener(urllib.request.ProxyHandler({}), NoRedirect())


def request_json(url, path, *, key=None, body=None, timeout=5):
    headers = {"Content-Type": "application/json"}
    if key is not None:
        headers["Authorization"] = "Bearer " + key
    data = None if body is None else json.dumps(body).encode()
    request = urllib.request.Request(url + path, data=data, headers=headers)
    with HTTP.open(request, timeout=timeout) as response:
        return json.load(response)


def check_authentication(url):
    try:
        request_json(url, "/v1/models")
    except urllib.error.HTTPError as error:
        if error.code == 401:
            return
        raise
    raise RuntimeError("Worker unexpectedly responded without a token.")


def wait_for_catalog(url, model, key, processes, log, timeout):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        if any(process.poll() is not None for process in processes):
            raise RuntimeError(f"A worker process stopped. See {log} for details.")
        try:
            data = request_json(url, "/v1/models", key=key)
        except urllib.error.HTTPError as error:
            if error.code not in (429, 502, 503, 504, 530):
                raise RuntimeError(
                    f"Model catalog returned HTTP {error.code}. See {log}."
                ) from None
        except (urllib.error.URLError, TimeoutError):
            pass
        else:
            if any(item["id"] == model for item in data["data"]):
                return
            raise RuntimeError(f"The worker catalog did not contain {model}.")
        time.sleep(2)
    raise TimeoutError(f"The model server did not become ready. See {log}.")


def check_structured_output(url, model, key):
    expected = {"ready": "worker_ready"}
    sample = {
        "model": model,
        "messages": [
            {"role": "system", "content": "Return a JSON object."},
            {"role": "user", "content": 'Return {"other": "unexpected"}.'},
        ],
        "temperature": 0,
        "max_tokens": 64,
        "response_format": {
            "type": "json_schema",
            "json_schema": {
                "name": "worker_probe",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {"ready": {"type": "string", "enum": ["worker_ready"]}},
                    "required": ["ready"],
                    "additionalProperties": False,
                },
            },
        },
    }
    response = request_json(url, "/v1/chat/completions", key=key, body=sample, timeout=90)
    if json.loads(response["choices"][0]["message"]["content"]) != expected:
        raise RuntimeError("The worker did not follow the requested JSON schema.")


def available_port():
    with socket.socket() as handle:
        handle.bind(("127.0.0.1", 0))
        return handle.getsockname()[1]


def stop_worker_tree(process):
    # vLLM tensor parallelism launches children in the worker process group.
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    try:
        process.wait(timeout=12)
    except subprocess.TimeoutExpired:
        pass
    try:
        os.killpg(process.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    process.wait(timeout=5)


def stop_llm_worker(stop_tunnel):
    tunnel = getattr(sys, "_research_assist_llm_tunnel", None)
    if tunnel is not None:
        stop_tunnel(tunnel)
    sys._research_assist_llm_tunnel = None
    for process in reversed(getattr(sys, "_research_assist_llm_processes", [])):
        stop_worker_tree(process)
    sys._research_assist_llm_processes = []


def worker_token():
    try:
        from kaggle_secrets import UserSecretsClient

        key = UserSecretsClient().get_secret("RESEARCH_GPU_API_KEY").strip()
    except Exception:
        raise RuntimeError(
            "Create RESEARCH_GPU_API_KEY in Kaggle Secrets and enable notebook access."
        ) from None
    if not re.fullmatch(r"[!-~]{32,256}", key):
        raise ValueError(
            "RESEARCH_GPU_API_KEY must contain 32–256 printable ASCII characters without spaces."
        )
    return key


def prepare_vllm(directory):
    environment = directory / f"venv-vllm-{VLLM_VERSION}"
    python = environment / "bin" / "python"
    if not python.is_file():
        venv.create(environment, with_pip=True)
    marker = environment / "research-assist-requirements.json"
    requirements = json.dumps(REQUIREMENTS)
    if not marker.is_file() or marker.read_text(encoding="utf-8") != requirements:
        print(f"Cài đặt vLLM {VLLM_VERSION} trong môi trường riêng…", flush=True)
        subprocess.run(
            [
                str(python),
                "-m",
                "pip",
                "install",
                "--disable-pip-version-check",
                "--no-cache-dir",
                "--only-binary=:all:",
                *REQUIREMENTS,
            ],
            check=True,
            timeout=1800,
        )
        marker.write_text(requirements, encoding="utf-8")
    return python


def run_llm_worker(profile, public_tunnel, gateway_source, *, start_tunnel, stop_tunnel):
    if platform.system() != "Linux" or platform.machine() != "x86_64":
        raise RuntimeError("Use a Kaggle Linux x86-64 notebook.")
    if sys.version_info[:2] not in {(3, 10), (3, 11), (3, 12)}:
        raise RuntimeError("This vLLM worker requires Python 3.10–3.12.")
    if profile not in PROFILES:
        raise ValueError('Set MODEL_PROFILE to "7b" or "32b".')
    key = worker_token()
    model, revision, tensor_parallel, context_length = PROFILES[profile]
    directory = Path("/kaggle/temp/research-assist-llm")
    directory.mkdir(parents=True, exist_ok=True)
    try:
        gpu_info = subprocess.run(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            capture_output=True,
            text=True,
            timeout=15,
            check=False,
        )
    except FileNotFoundError:
        raise RuntimeError("Enable an NVIDIA GPU accelerator in Kaggle.") from None
    if gpu_info.returncode or len(gpu_info.stdout.splitlines()) < tensor_parallel:
        raise RuntimeError(f"Enable an Accelerator with at least {tensor_parallel} visible GPU(s).")
    print(f"GPU: {', '.join(gpu_info.stdout.splitlines())}")
    stop_llm_worker(stop_tunnel)
    if not getattr(sys, "_research_assist_llm_exit_registered", False):
        atexit.register(stop_llm_worker, stop_tunnel)
        sys._research_assist_llm_exit_registered = True

    python = prepare_vllm(directory)
    env = os.environ.copy()
    env.pop("RESEARCH_GPU_API_KEY", None)
    env.update(
        VLLM_USE_V1="0",
        VLLM_ATTENTION_BACKEND="XFORMERS",
        VLLM_NO_USAGE_STATS="1",
        HF_HOME=str(directory / "hf"),
        PYTHONNOUSERSITE="1",
    )
    cuda_check = (
        "import torch, sys; n = int(sys.argv[1]); "
        "assert torch.cuda.device_count() >= n, 'Not enough CUDA devices'; "
        "assert all(torch.cuda.get_device_capability(i) >= (7,5) for i in range(n)), "
        "'AWQ requires SM 7.5 or newer'; "
        "assert all(float(torch.ones(1, device=f'cuda:{i}').item()) == 1 for i in range(n))"
    )
    subprocess.run(
        [str(python), "-c", cuda_check, str(tensor_parallel)], env=env, check=True, timeout=120
    )
    upstream = available_port()
    port = available_port()
    while port == upstream:
        port = available_port()
    local_url = f"http://127.0.0.1:{port}"
    gateway_path = directory / "gateway.py"
    gateway_path.write_text(gateway_source, encoding="utf-8")
    log_path = directory / "worker.log"
    try:
        print(f"Đang tải và khởi động {model} (TP={tensor_parallel})…", flush=True)
        command = [
            str(python),
            "-m",
            "vllm.entrypoints.openai.api_server",
            "--model",
            model,
            "--revision",
            revision,
            "--load-format",
            "safetensors",
            "--host",
            "127.0.0.1",
            "--port",
            str(upstream),
            "--tensor-parallel-size",
            str(tensor_parallel),
            "--distributed-executor-backend",
            "mp",
            "--dtype",
            "float16",
            "--quantization",
            "awq",
            "--enforce-eager",
            "--max-model-len",
            str(context_length),
            "--max-num-seqs",
            "2",
            "--gpu-memory-utilization",
            "0.90" if tensor_parallel == 1 else "0.94",
            "--guided-decoding-backend",
            "xgrammar",
            "--generation-config",
            "vllm",
            "--disable-log-requests",
        ]
        if tensor_parallel > 1:
            command += ["--disable-custom-all-reduce"]
        processes = sys._research_assist_llm_processes
        with log_path.open("w", encoding="utf-8") as log:
            server = subprocess.Popen(
                command, env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True
            )
            processes.append(server)
            gateway = subprocess.Popen(
                [str(python), str(gateway_path), str(upstream), str(port)],
                env={**env, "RESEARCH_GPU_API_KEY": key},
                stdout=log,
                stderr=subprocess.STDOUT,
                start_new_session=True,
            )
            processes.append(gateway)
        print(f"Xem tiến trình tải model tại {log_path}")
        wait_for_catalog(local_url, model, key, processes, log_path, timeout=1800)
        check_authentication(local_url)
        check_structured_output(local_url, model, key)
        print("Worker đã xác nhận API Bearer và phản hồi JSON theo schema.")

        url = local_url
        if public_tunnel:
            tunnel, url = start_tunnel(port, directory / "tunnel")
            sys._research_assist_llm_tunnel = tunnel
            wait_for_catalog(url, model, key, [*processes, tunnel], log_path, timeout=70)
            check_authentication(url)
        print(f"Provider = OpenAI-compatible\nEndpoint /v1 = {url}/v1\nModel = {model}")
        print("API key: dùng giá trị RESEARCH_GPU_API_KEY đã lưu trong Kaggle Secrets.")
    except BaseException:
        stop_llm_worker(stop_tunnel)
        raise

In [ ]:
GATEWAY_SOURCE = '"""Expose only the authenticated endpoints needed by Research Assist."""\n\nimport json\nimport os\nimport secrets\nfrom contextlib import asynccontextmanager\n\nimport httpx\nfrom fastapi import FastAPI, HTTPException, Request\nfrom fastapi.responses import Response\n\nMAX_REQUEST_BYTES = 1024 * 1024\nCHAT_FIELDS = {"model", "messages", "temperature", "max_tokens", "response_format"}\n\n\ndef create_gateway(upstream_port: int, key: str) -> FastAPI:\n    if not key:\n        raise ValueError("A worker token is required.")\n    authorization = ("Bearer " + key).encode()\n\n    @asynccontextmanager\n    async def lifespan(app: FastAPI):\n        async with httpx.AsyncClient(\n            base_url=f"http://127.0.0.1:{upstream_port}",\n            timeout=httpx.Timeout(85, connect=5),\n            trust_env=False,\n        ) as client:\n            app.state.serving_client = client\n            yield\n\n    app = FastAPI(openapi_url=None, docs_url=None, redoc_url=None, lifespan=lifespan)\n\n    def require_key(request: Request) -> None:\n        candidate = request.headers.get("Authorization", "").encode()\n        if not secrets.compare_digest(candidate, authorization):\n            raise HTTPException(401, "Worker token is missing or invalid.")\n\n    async def forward(path: str, method: str, body: dict | None = None) -> Response:\n        try:\n            reply = await app.state.serving_client.request(method, path, json=body)\n        except httpx.TimeoutException:\n            raise HTTPException(504, "Model response timed out.") from None\n        except httpx.RequestError:\n            raise HTTPException(503, "The model server is not available.") from None\n        return Response(\n            reply.content,\n            status_code=reply.status_code,\n            media_type="application/json",\n            headers={"Cache-Control": "no-store"},\n        )\n\n    @app.get("/v1/models")\n    async def models(request: Request) -> Response:\n        require_key(request)\n        return await forward("/v1/models", "GET")\n\n    @app.post("/v1/chat/completions")\n    async def chat_completions(request: Request) -> Response:\n        require_key(request)\n        raw = bytearray()\n        async for chunk in request.stream():\n            if len(raw) + len(chunk) > MAX_REQUEST_BYTES:\n                raise HTTPException(413, "Request is too large.")\n            raw.extend(chunk)\n        try:\n            body = json.loads(raw)\n        except (ValueError, RecursionError):\n            raise HTTPException(400, "Invalid JSON.") from None\n        if not isinstance(body, dict) or body.keys() - CHAT_FIELDS:\n            raise HTTPException(400, "Only Research Assist chat fields are accepted.")\n        messages = body.get("messages")\n        if (\n            not isinstance(messages, list)\n            or not 1 <= len(messages) <= 32\n            or any(\n                not isinstance(message, dict)\n                or message.keys() != {"role", "content"}\n                or message["role"] not in ("system", "user", "assistant")\n                or not isinstance(message["content"], str)\n                for message in messages\n            )\n        ):\n            raise HTTPException(400, "Only plain-text chat messages are accepted.")\n        return await forward("/v1/chat/completions", "POST", body)\n\n    return app\n\n\nif __name__ == "__main__":\n    import sys\n\n    import uvicorn\n\n    upstream, port = map(int, sys.argv[1:3])\n    gateway = create_gateway(upstream, os.environ.pop("RESEARCH_GPU_API_KEY"))\n    uvicorn.run(gateway, host="127.0.0.1", port=port, access_log=False)\n'

## Khởi động worker

In [ ]:
if not START_LLM_WORKER:
    print("Đặt START_LLM_WORKER = True và chạy lại từ ô cấu hình để mở worker.")
else:
    run_llm_worker(
        MODEL_PROFILE, OPEN_PUBLIC_TUNNEL, GATEWAY_SOURCE,
        start_tunnel=start_quick_tunnel, stop_tunnel=stop_process,
    )
    if KEEP_WORKER_RUNNING:
        print("Worker đang chạy. Ngắt ô này để dừng.", flush=True)
        try:
            children = list(sys._research_assist_llm_processes)
            if sys._research_assist_llm_tunnel is not None:
                children.append(sys._research_assist_llm_tunnel)
            while all(process.poll() is None for process in children):
                time.sleep(2)
            print("Một tiến trình worker đã dừng. Xem log ở đường dẫn in phía trên.")
        except KeyboardInterrupt:
            print("Đang dừng worker…")
        finally:
            stop_llm_worker(stop_process)

## Dừng worker
Bật `STOP_LLM_WORKER` và chạy ô dưới khi cần dừng worker.

In [ ]:
STOP_LLM_WORKER = False
if STOP_LLM_WORKER:
    stop_llm_worker(stop_process)